In [1]:
%pip install qiskit qiskit_aer qiskit_machine_learning torch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/9.9 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 4.7/9.9 MB 26.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 8.7/9.9 MB 22.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 21.1 MB/s eta 0:00:00


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
import pandas as pd

n_samples = 2000

df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["has_fire"].values.astype(float)  # binary 0/1

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

print(f"Train: {x_train_a.shape}, Test: {x_test_a.shape}")
print(f"Fire rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

Train: (1800, 9), Test: (200, 9)
Fire rate — train: 1.1%  test: 0.5%


In [4]:
n_features = x_train_a.shape[1]
n_qubits = n_features

x_params = ParameterVector("x", n_features)
reps = 3
n_theta = (n_qubits)
theta_params = ParameterVector("θ", length=n_qubits * 2 * reps)

qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.ry(x_params[i], i)

t = 0
for r in range(reps):
    for q in range(n_qubits):
        qc.ry(theta_params[t], q); t += 1
        qc.rz(theta_params[t], q); t += 1

    for q in range(n_qubits):
        for r in range(q+1, n_qubits):
            qc.cx(q, r)

qc.decompose().draw("text")

global phase: (-0.5)*θ[1] - 0.5*θ[3] - 0.5*θ[5] - 0.5*θ[7] - 0.5*θ[9] - 0.5*θ[11] - 0.5*θ[13] - 0.5*θ[15] - 0.5*θ[17] - 0.5*θ[19] - 0.5*θ[21] - 0.5*θ[23] - 0.5*θ[25] - 0.5*θ[27] - 0.5*θ[29] - 0.5*θ[31] - 0.5*θ[33] - 0.5*θ[35] - 0.5*θ[37] - 0.5*θ[39] - 0.5*θ[41] - 0.5*θ[43] - 0.5*θ[45] - 0.5*θ[47] - 0.5*θ[49] - 0.5*θ[51] - 0.5*θ[53]
     ┌─────────────┐┌─────────────┐ ┌─────────┐                               »
q_0: ┤ R(x[0],π/2) ├┤ R(θ[0],π/2) ├─┤ P(θ[1]) ├───■────■─────────■─────────■──»
     ├─────────────┤├─────────────┤ ├─────────┤ ┌─┴─┐  │         │         │  »
q_1: ┤ R(x[1],π/2) ├┤ R(θ[2],π/2) ├─┤ P(θ[3]) ├─┤ X ├──┼────■────┼────■────┼──»
     ├─────────────┤├─────────────┤ ├─────────┤ └───┘┌─┴─┐┌─┴─┐  │    │    │  »
q_2: ┤ R(x[2],π/2) ├┤ R(θ[4],π/2) ├─┤ P(θ[5]) ├──────┤ X ├┤ X ├──┼────┼────┼──»
     ├─────────────┤├─────────────┤ ├─────────┤      └───┘└───┘┌─┴─┐┌─┴─┐  │  »
q_3: ┤ R(x[3],π/2) ├┤ R(θ[6],π/2) ├─┤ P(θ[7]) ├────────────────┤ X ├┤ X ├──┼──»
     ├─────────────┤├─────────────┤ ├─────────┤                └───┘└───┘┌─┴─┐»
q_4: ┤ R(x[4],π/2) ├┤ R(θ[8],π/2) ├─┤ P(θ[9]) ├──────────────────────────┤ X ├»
     ├─────────────┤├─────────────┴┐├─────────┴┐                         └───┘»
q_5: ┤ R(x[5],π/2) ├┤ R(θ[10],π/2) ├┤ P(θ[11]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_6: ┤ R(x[6],π/2) ├┤ R(θ[12],π/2) ├┤ P(θ[13]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_7: ┤ R(x[7],π/2) ├┤ R(θ[14],π/2) ├┤ P(θ[15]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_8: ┤ R(x[8],π/2) ├┤ R(θ[16],π/2) ├┤ P(θ[17]) ├──────────────────────────────»
     └─────────────┘└──────────────┘└──────────┘                              »
«                                                                           »
«q_0: ────────────■──────────────■───────────────────■───────────────────■──»
«                 │              │                   │                   │  »
«q_1: ───────■────┼─────────■────┼──────────────■────┼──────────────■────┼──»
«            │    │         │    │              │    │              │    │  »
«q_2: ──■────┼────┼────■────┼────┼─────────■────┼────┼─────────■────┼────┼──»
«     ┌─┴─┐  │    │    │    │    │         │    │    │         │    │    │  »
«q_3: ┤ X ├──┼────┼────┼────┼────┼────■────┼────┼────┼────■────┼────┼────┼──»
«     └───┘┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │    │    │    │    │  »
«q_4: ─────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼────┼────┼────┼────┼──»
«          └───┘┌─┴─┐└───┘┌─┴─┐  │  └───┘┌─┴─┐  │    │  ┌─┴─┐  │    │    │  »
«q_5: ──────────┤ X ├─────┤ X ├──┼───────┤ X ├──┼────┼──┤ X ├──┼────┼────┼──»
«               └───┘     └───┘┌─┴─┐     └───┘┌─┴─┐  │  └───┘┌─┴─┐  │    │  »
«q_6: ─────────────────────────┤ X ├──────────┤ X ├──┼───────┤ X ├──┼────┼──»
«                              └───┘          └───┘┌─┴─┐     └───┘┌─┴─┐  │  »
«q_7: ─────────────────────────────────────────────┤ X ├──────────┤ X ├──┼──»
«                                                  └───┘          └───┘┌─┴─┐»
«q_8: ─────────────────────────────────────────────────────────────────┤ X ├»
«                                                                      └───┘»
«     ┌──────────────┐┌──────────┐                                           »
«q_0: ┤ R(θ[18],π/2) ├┤ P(θ[19]) ├────────────────────────────────────────■──»
«     └──────────────┘└──────────┘          ┌──────────────┐┌──────────┐┌─┴─┐»
«q_1: ───────────────────────────────────■──┤ R(θ[20],π/2) ├┤ P(θ[21]) ├┤ X ├»
«                                        │  └──────────────┘└──────────┘└───┘»
«q_2: ──────────────────────────────■────┼────────────────────────────────■──»
«                                   │    │                                │  »
«q_3: ─────────────────────■────────┼────┼───────────────────────■────────┼──»
«                          │    

In [5]:
observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])

estimator = Estimator()

qnn = EstimatorQNN(
    circuit=qc,
    estimator=estimator,
    observables=observable,
    input_params=list(x_params),
    weight_params=list(theta_params),
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [6]:
import csv

torch.manual_seed(0)

model = TorchConnector(qnn)

loss_fn = nn.BCEWithLogitsLoss()  # correct for binary 0/1 labels
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

xtr = torch.tensor(x_train_a, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

param_names = [f"theta_{i}" for i in range(len(list(model.parameters())[0]))]
csv_path = "params_per_epoch.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "loss"] + param_names)

    for epoch in range(20):
        optimizer.zero_grad()
        yhat = model(xtr)
        loss = loss_fn(yhat, ytr)
        loss.backward()
        optimizer.step()

        params = list(model.parameters())[0].detach().numpy().tolist()
        writer.writerow([epoch + 1, loss.item()] + params)
        print(f"epoch={epoch+1:3d}, loss={loss.item():.6f}")

print(f"\nParams saved to {csv_path}")

epoch=  1, loss=0.728339


epoch=  2, loss=0.716745


epoch=  3, loss=0.703900


epoch=  4, loss=0.688060


epoch=  5, loss=0.670439


...

epoch= 16, loss=0.481618


epoch= 17, loss=0.473564


epoch= 18, loss=0.467170


epoch= 19, loss=0.462404


epoch= 20, loss=0.458918

Params saved to params_per_epoch.csv


In [7]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

xte = torch.tensor(x_test_a, dtype=torch.float32)
with torch.no_grad():
    logits = model(xte).numpy().ravel()

y_prob = 1 / (1 + np.exp(-logits))  # sigmoid → probabilities
y_pred = (y_prob >= 0.5).astype(int)

print("MAE: ", mean_absolute_error(y_test, y_prob))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_prob)))
print("R2:  ", r2_score(y_test, y_prob))

MAE:  0.3567784714698792
RMSE: 0.36455862041910014
R2:   -25.71416838630703


In [8]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# y_prob and y_pred already computed in previous cell
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

n_classes = len(np.unique(y_test))
auc = roc_auc_score(y_test, y_prob) if n_classes == 2 else float("nan")

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"AUC_ROC:   {auc:.4f}")

Precision: 0.0000
Recall:    0.0000
F1:        0.0000
AUC_ROC:   0.8040
